In [15]:
import hashlib
from pathlib import Path
import pandas as pd

audio_root = Path("/content/Voice_Dataset")
csv_path = "/content/parkinsons_voice_features.csv"

# Ensure the source CSV exists
if not Path(csv_path).exists():
    raise FileNotFoundError(f"Source {csv_path} not found. Please run the feature extraction cell first!")

df = pd.read_csv(csv_path)
df["filename"] = df["filename"].astype(str)

# Group audio files by exact content if the directory is available
unique_filenames = []
if audio_root.exists():
    audio_groups = {}
    for path in sorted(audio_root.rglob("*.wav")):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        audio_groups.setdefault(digest, []).append(path.name)

    # Keep one file from each exact-duplicate group
    for digest, filenames in audio_groups.items():
        filenames_in_csv = [name for name in filenames if name in set(df["filename"])]
        if not filenames_in_csv:
            continue
        group_rows = df[df["filename"].isin(filenames_in_csv)]
        if group_rows["Status"].nunique() != 1:
            raise ValueError(f"Conflicting labels in duplicate group: {filenames_in_csv}")
        unique_filenames.append(sorted(filenames_in_csv)[0])

    unique_df = df[df["filename"].isin(unique_filenames)].drop_duplicates(subset="filename").copy()
else:
    print("Voice_Dataset folder not found. Bypassing content deduplication and keeping all loaded features.")
    unique_df = df.copy()

unique_df.to_csv("/content/parkinsons_voice_features_unique.csv", index=False)
print("Original rows:", len(df))
print("Unique rows saved:", len(unique_df))
print("Saved: /content/parkinsons_voice_features_unique.csv")

Voice_Dataset folder not found. Bypassing content deduplication and keeping all loaded features.
Original rows: 1134
Unique rows saved: 1134
Saved: /content/parkinsons_voice_features_unique.csv


In [ ]:
import shutil
from pathlib import Path

# Fix the uploaded filename mismatch
expected_path = Path("/content/parkinsons_voice_features.csv")
uploaded_path = Path("/content/parkinsons_voice_features (1).csv")

if uploaded_path.exists() and not expected_path.exists():
    shutil.copy(uploaded_path, expected_path)
    print(f"Successfully copied {uploaded_path.name} to {expected_path.name}!")
else:
    print("Path check:")
    print("- Expected file exists:", expected_path.exists())
    print("- Uploaded file (1) exists:", uploaded_path.exists())

In [ ]:
import hashlib
from pathlib import Path
import pandas as pd

audio_root = Path("/content/Voice_Dataset")
csv_path = "/content/parkinsons_voice_features.csv"

# Ensure the source CSV exists
if not Path(csv_path).exists():
    raise FileNotFoundError(f"Source {csv_path} not found. Please ensure the features CSV is uploaded!")

df = pd.read_csv(csv_path)
df["filename"] = df["filename"].astype(str)

# Group audio files by exact content if the audio folder exists
unique_filenames = []
if audio_root.exists():
    audio_groups = {}
    for path in sorted(audio_root.rglob("*.wav")):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        audio_groups.setdefault(digest, []).append(path.name)

    # Keep one file from each exact-duplicate group
    for digest, filenames in audio_groups.items():
        filenames_in_csv = [name for name in filenames if name in set(df["filename"])]
        if not filenames_in_csv:
            continue
        group_rows = df[df["filename"].isin(filenames_in_csv)]
        if group_rows["Status"].nunique() != 1:
            raise ValueError(f"Conflicting labels in duplicate group: {filenames_in_csv}")
        unique_filenames.append(sorted(filenames_in_csv)[0])

    unique_df = df[df["filename"].isin(unique_filenames)].drop_duplicates(subset="filename").copy()
else:
    print("Audio root folder not found, keeping original uploaded features for analysis.")
    unique_df = df.copy()

unique_df.to_csv("/content/parkinsons_voice_features_unique.csv", index=False)
print("Original rows:", len(df))
print("Unique rows saved:", len(unique_df))
print("Saved: /content/parkinsons_voice_features_unique.csv")

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

unique_csv_path = "/content/parkinsons_voice_features_unique.csv"

# Load the unique dataset
df = pd.read_csv(unique_csv_path)

# Define feature groups
voice_quality = [
    "Jitter_local", "Shimmer_local",
    "HNR_dB", "Voice_Ratio"
]

spectral = (
    [f"MFCC{i}" for i in range(13)] +
    ["Spectral_Centroid", "Spectral_Bandwidth",
     "Zero_Crossing_Rate"]
)

prosodic = [
    "Pitch_mean_Hz", "Pitch_std_Hz",
    "Speech_Rate_proxy"
]

# Safely filter available features
available_cols = set(df.columns)
voice_quality = [f for f in voice_quality if f in available_cols]
spectral = [f for f in spectral if f in available_cols]
prosodic = [f for f in prosodic if f in available_cols]

feature_groups = {
    "Voice Quality": voice_quality,
    "Spectral": spectral,
    "Prosodic": prosodic,
    "Combined": voice_quality + spectral + prosodic
}

y = df["Status"]

cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

results = []

for group_name, features in feature_groups.items():
    if not features:
        continue
    X = df[features]

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        model, X, y, cv=cv,
        scoring=["accuracy", "precision", "recall", "f1"]
    )

    results.append({
        "Feature Group": group_name,
        "Accuracy (%)": round(
            scores["test_accuracy"].mean() * 100, 2
        ),
        "Precision (%)": round(
            scores["test_precision"].mean() * 100, 2
        ),
        "Recall (%)": round(
            scores["test_recall"].mean() * 100, 2
        ),
        "F1 Score (%)": round(
            scores["test_f1"].mean() * 100, 2
        )
    })

results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv(
    "/content/feature_group_comparison.csv",
    index=False
)
print("Saved feature group comparison successfully!")

In [12]:
import shutil
from pathlib import Path

# Correcting the file naming difference from your upload
expected_path = Path("/content/parkinsons_voice_features.csv")
uploaded_path = Path("/content/parkinsons_voice_features (1).csv")

if uploaded_path.exists():
    shutil.copy(uploaded_path, expected_path)
    print(f"Success: Copied {uploaded_path.name} to {expected_path.name}!")
else:
    print(f"File {uploaded_path} not found. Checking if expected file already exists...")
    if expected_path.exists():
        print("The expected file already exists at /content/parkinsons_voice_features.csv!")
    else:
        print("Error: No uploaded file was found in /content/!")

Success: Copied parkinsons_voice_features (1).csv to parkinsons_voice_features.csv!


In [13]:
import hashlib
from pathlib import Path
import pandas as pd

audio_root = Path("/content/Voice_Dataset")
csv_path = "/content/parkinsons_voice_features.csv"

# Ensure the source CSV exists
if not Path(csv_path).exists():
    raise FileNotFoundError(f"Source {csv_path} not found. Please ensure the CSV has been copied correctly in the previous step.")

df = pd.read_csv(csv_path)
df["filename"] = df["filename"].astype(str)

unique_filenames = []
if audio_root.exists():
    audio_groups = {}
    for path in sorted(audio_root.rglob("*.wav")):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        audio_groups.setdefault(digest, []).append(path.name)

    for digest, filenames in audio_groups.items():
        filenames_in_csv = [name for name in filenames if name in set(df["filename"])]
        if not filenames_in_csv:
            continue
        group_rows = df[df["filename"].isin(filenames_in_csv)]
        if group_rows["Status"].nunique() != 1:
            raise ValueError(f"Conflicting labels in duplicate group: {filenames_in_csv}")
        unique_filenames.append(sorted(filenames_in_csv)[0])

    unique_df = df[df["filename"].isin(unique_filenames)].drop_duplicates(subset="filename").copy()
else:
    print("Voice_Dataset folder not found. Running dataset copy directly.")
    unique_df = df.copy()

unique_df.to_csv("/content/parkinsons_voice_features_unique.csv", index=False)
print("Original dataset rows:", len(df))
print("Deduplicated dataset rows:", len(unique_df))
print("Saved successfully to: /content/parkinsons_voice_features_unique.csv")

Voice_Dataset folder not found. Running dataset copy directly.
Original dataset rows: 1134
Deduplicated dataset rows: 1134
Saved successfully to: /content/parkinsons_voice_features_unique.csv


In [14]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

unique_csv_path = "/content/parkinsons_voice_features_unique.csv"

if not os.path.exists(unique_csv_path):
    raise FileNotFoundError(f"Could not find '{unique_csv_path}'. Please run the previous cell first.")

# Load the unique dataset
df = pd.read_csv(unique_csv_path)

# Define feature groups
voice_quality = [
    "Jitter_local", "Shimmer_local",
    "HNR_dB", "Voice_Ratio"
]

spectral = (
    [f"MFCC{i}" for i in range(13)] +
    ["Spectral_Centroid", "Spectral_Bandwidth",
     "Zero_Crossing_Rate"]
)

prosodic = [
    "Pitch_mean_Hz", "Pitch_std_Hz",
    "Speech_Rate_proxy"
]

# Safely filter available features
available_cols = set(df.columns)
voice_quality = [f for f in voice_quality if f in available_cols]
spectral = [f for f in spectral if f in available_cols]
prosodic = [f for f in prosodic if f in available_cols]

feature_groups = {
    "Voice Quality": voice_quality,
    "Spectral": spectral,
    "Prosodic": prosodic,
    "Combined": voice_quality + spectral + prosodic
}

y = df["Status"]

cv = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=42
)

results = []

for group_name, features in feature_groups.items():
    if not features:
        continue
    X = df[features]

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42
        ))
    ])

    scores = cross_validate(
        model, X, y, cv=cv,
        scoring=["accuracy", "precision", "recall", "f1"]
    )

    results.append({
        "Feature Group": group_name,
        "Accuracy (%)": round(
            scores["test_accuracy"].mean() * 100, 2
        ),
        "Precision (%)": round(
            scores["test_precision"].mean() * 100, 2
        ),
        "Recall (%)": round(
            scores["test_recall"].mean() * 100, 2
        ),
        "F1 Score (%)": round(
            scores["test_f1"].mean() * 100, 2
        )
    })

results_df = pd.DataFrame(results)
display(results_df)

results_df.to_csv(
    "/content/feature_group_comparison.csv",
    index=False
)
print("Saved feature group comparison successfully!")

,Feature Group,Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Voice Quality,96.83,97.50,96.07,96.77
1,Spectral,96.83,96.81,96.79,96.78
2,Prosodic,97.18,96.56,97.86,97.18
3,Combined,99.12,98.95,99.29,99.11


Saved feature group comparison successfully!


In [16]:
!pip install librosa soundfile praat-parselmouth pandas


In [17]:

!pip install librosa soundfile praat-parselmouth pandas scikit-learn



In [18]:

import librosa
import soundfile as sf
import parselmouth
import pandas as pd
import sklearn

print("All libraries imported successfully!")
print("Librosa version:", librosa.__version__)
print("Scikit-learn version:", sklearn.__version__)


All libraries imported successfully!
Librosa version: 0.11.0
Scikit-learn version: 1.6.1


In [19]:

from google.colab import files

uploaded = files.upload()


Saving archive (1).zip to archive (1).zip


In [35]:
import pandas as pd
import numpy as np
from pathlib import Path

# Pointing to the correct, fully aligned unique dataset generated earlier
file_path = "/content/parkinsons_voice_features_unique.csv"

if not Path(file_path).exists():
    # Fallback to the raw features CSV if unique is missing
    file_path = "/content/parkinsons_voice_features.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:", df.isnull().sum().sum())

print("\nStatus distribution:")
print(df["Status"].value_counts())

Dataset loaded successfully!
Dataset shape: (1134, 26)

Column names:
['filename', 'duration_sec', 'Pitch_mean_Hz', 'Pitch_std_Hz', 'Jitter_local', 'Shimmer_local', 'HNR_dB', 'Voice_Ratio', 'Speech_Rate_proxy', 'Spectral_Centroid', 'Spectral_Bandwidth', 'Zero_Crossing_Rate', 'Status', 'MFCC0', 'MFCC1', 'MFCC2', 'MFCC3', 'MFCC4', 'MFCC5', 'MFCC6', 'MFCC7', 'MFCC8', 'MFCC9', 'MFCC10', 'MFCC11', 'MFCC12']

First 5 rows:


,filename,duration_sec,Pitch_mean_Hz,Pitch_std_Hz,Jitter_local,Shimmer_local,HNR_dB,Voice_Ratio,Speech_Rate_proxy,Spectral_Centroid,...,MFCC3,MFCC4,MFCC5,MFCC6,MFCC7,MFCC8,MFCC9,MFCC10,MFCC11,MFCC12
0,healthy_000.wav,3.738875,131.555537,1.870241,0.004138,0.038810,20.302154,0.994595,3.744442,1104.893589,...,-37.945095,-5.174286,-7.978308,-11.592722,7.337579,9.517365,-9.009621,-21.090189,-7.455282,-6.077338
1,healthy_001.wav,3.738875,131.555537,1.870241,0.004132,0.038669,20.133017,0.994595,3.476982,1113.599827,...,-37.364166,-4.286234,-7.800795,-10.694014,7.303071,9.929105,-8.755672,-20.469978,-7.607912,-5.230393
2,healthy_002.wav,3.738875,131.555537,1.870241,0.004138,0.038810,20.302171,0.994595,3.744442,1104.891320,...,-37.944454,-5.173617,-7.977430,-11.592191,7.338422,9.517933,-9.009238,-21.089542,-7.455106,-6.076827
3,healthy_003.wav,3.738875,117.287689,1.695553,0.002310,0.095814,15.489594,0.991892,1.872221,974.037124,...,-17.635220,-37.965416,21.846725,-36.532772,19.399149,-11.420612,28.773481,-27.703768,2.464287,-26.210688
4,healthy_004.wav,3.738875,147.665658,2.068247,0.003124,0.103435,14.282871,0.994595,2.674601,1172.725617,...,-24.638401,-3.243519,-9.444654,-3.337813,14.391733,-7.669085,-22.883625,-11.609543,0.177867,0.819698



Missing values: 0

Status distribution:
Status
0    574
1    560
Name: count, dtype: int64


In [30]:

print("Dataset shape:", df.shape)

print("\nAll columns:")
for col in df.columns:
    print(col)

print("\nTarget distribution:")
print(df["Status"].value_counts())

print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (1134, 26)

All columns:
filename
duration_sec
Pitch_mean_Hz
Pitch_std_Hz
Jitter_local
Shimmer_local
HNR_dB
Voice_Ratio
Speech_Rate_proxy
Spectral_Centroid
Spectral_Bandwidth
Zero_Crossing_Rate
Status
MFCC0
MFCC1
MFCC2
MFCC3
MFCC4
MFCC5
MFCC6
MFCC7
MFCC8
MFCC9
MFCC10
MFCC11
MFCC12

Target distribution:
Status
0    574
1    560
Name: count, dtype: int64

First 5 rows:


,filename,duration_sec,Pitch_mean_Hz,Pitch_std_Hz,Jitter_local,Shimmer_local,HNR_dB,Voice_Ratio,Speech_Rate_proxy,Spectral_Centroid,...,MFCC3,MFCC4,MFCC5,MFCC6,MFCC7,MFCC8,MFCC9,MFCC10,MFCC11,MFCC12
0,healthy_000.wav,3.738875,131.555537,1.870241,0.004138,0.038810,20.302154,0.994595,3.744442,1104.893589,...,-37.945095,-5.174286,-7.978308,-11.592722,7.337579,9.517365,-9.009621,-21.090189,-7.455282,-6.077338
1,healthy_001.wav,3.738875,131.555537,1.870241,0.004132,0.038669,20.133017,0.994595,3.476982,1113.599827,...,-37.364166,-4.286234,-7.800795,-10.694014,7.303071,9.929105,-8.755672,-20.469978,-7.607912,-5.230393
2,healthy_002.wav,3.738875,131.555537,1.870241,0.004138,0.038810,20.302171,0.994595,3.744442,1104.891320,...,-37.944454,-5.173617,-7.977430,-11.592191,7.338422,9.517933,-9.009238,-21.089542,-7.455106,-6.076827
3,healthy_003.wav,3.738875,117.287689,1.695553,0.002310,0.095814,15.489594,0.991892,1.872221,974.037124,...,-17.635220,-37.965416,21.846725,-36.532772,19.399149,-11.420612,28.773481,-27.703768,2.464287,-26.210688
4,healthy_004.wav,3.738875,147.665658,2.068247,0.003124,0.103435,14.282871,0.994595,2.674601,1172.725617,...,-24.638401,-3.243519,-9.444654,-3.337813,14.391733,-7.669085,-22.883625,-11.609543,0.177867,0.819698


In [27]:

print("Total columns:", len(df.columns))

for col in df.columns:
    print(col)


Total columns: 26
filename
duration_sec
Pitch_mean_Hz
Pitch_std_Hz
Jitter_local
Shimmer_local
HNR_dB
Voice_Ratio
Speech_Rate_proxy
Spectral_Centroid
Spectral_Bandwidth
Zero_Crossing_Rate
Status
MFCC0
MFCC1
MFCC2
MFCC3
MFCC4
MFCC5
MFCC6
MFCC7
MFCC8
MFCC9
MFCC10
MFCC11
MFCC12


In [42]:
# Show file identifiers and targets
print(df[["filename", "Status"]].head(10))

          filename  Status
0  healthy_000.wav       0
1  healthy_001.wav       0
2  healthy_002.wav       0
3  healthy_003.wav       0
4  healthy_004.wav       0
5  healthy_005.wav       0
6  healthy_006.wav       0
7  healthy_007.wav       0
8  healthy_008.wav       0
9  healthy_009.wav       0


In [43]:
print("Dataset shape:", df.shape)

print("\nUnique recording files:")
print(df["filename"].nunique())

print("\nStatus distribution:")
print(df["Status"].value_counts())

print("\nSample filenames:")
print(df["filename"].head(10).tolist())

Dataset shape: (1134, 26)

Unique recording files:
1134

Status distribution:
Status
0    574
1    560
Name: count, dtype: int64

Sample filenames:
['healthy_000.wav', 'healthy_001.wav', 'healthy_002.wav', 'healthy_003.wav', 'healthy_004.wav', 'healthy_005.wav', 'healthy_006.wav', 'healthy_007.wav', 'healthy_008.wav', 'healthy_009.wav']


In [39]:
from google.colab import files

uploaded = files.upload()

Saving archive (1).zip to archive (1) (1).zip


In [40]:
import zipfile
import os

zip_path = "/content/archive (1).zip"
extract_path = "/content/Voice_Dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

print("Extraction completed!")

Extraction completed!


In [44]:
import os

extract_path = "/content/Voice_Dataset"

# Walk through the folder structure and count files
print("Folder Structure and File Counts:")
for root, dirs, files in os.walk(extract_path):
    # Filter out hidden files
    visible_files = [f for f in files if not f.startswith('.')]
    if len(dirs) > 0 or len(visible_files) > 0:
        level = root.replace(extract_path, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/ ({len(visible_files)} files)")
        # Print first 3 sample files if present
        if visible_files:
            for sample in sorted(visible_files)[:3]:
                print(f"{indent}  - {sample}")
            if len(visible_files) > 3:
                print(f"{indent}  - ... and {len(visible_files) - 3} more files")

Folder Structure and File Counts:
Voice_Dataset/ (0 files)
    Voice_Dataset/ (0 files)
        Parkinsons/ (560 files)
          - parkinsons_000.wav
          - parkinsons_001.wav
          - parkinsons_002.wav
          - ... and 557 more files
        Healthy/ (574 files)
          - healthy_000.wav
          - healthy_001.wav
          - healthy_002.wav
          - ... and 571 more files


In [45]:
display(df.head())

,filename,duration_sec,Pitch_mean_Hz,Pitch_std_Hz,Jitter_local,Shimmer_local,HNR_dB,Voice_Ratio,Speech_Rate_proxy,Spectral_Centroid,...,MFCC3,MFCC4,MFCC5,MFCC6,MFCC7,MFCC8,MFCC9,MFCC10,MFCC11,MFCC12
0,healthy_000.wav,3.738875,131.555537,1.870241,0.004138,0.038810,20.302154,0.994595,3.744442,1104.893589,...,-37.945095,-5.174286,-7.978308,-11.592722,7.337579,9.517365,-9.009621,-21.090189,-7.455282,-6.077338
1,healthy_001.wav,3.738875,131.555537,1.870241,0.004132,0.038669,20.133017,0.994595,3.476982,1113.599827,...,-37.364166,-4.286234,-7.800795,-10.694014,7.303071,9.929105,-8.755672,-20.469978,-7.607912,-5.230393
2,healthy_002.wav,3.738875,131.555537,1.870241,0.004138,0.038810,20.302171,0.994595,3.744442,1104.891320,...,-37.944454,-5.173617,-7.977430,-11.592191,7.338422,9.517933,-9.009238,-21.089542,-7.455106,-6.076827
3,healthy_003.wav,3.738875,117.287689,1.695553,0.002310,0.095814,15.489594,0.991892,1.872221,974.037124,...,-17.635220,-37.965416,21.846725,-36.532772,19.399149,-11.420612,28.773481,-27.703768,2.464287,-26.210688
4,healthy_004.wav,3.738875,147.665658,2.068247,0.003124,0.103435,14.282871,0.994595,2.674601,1172.725617,...,-24.638401,-3.243519,-9.444654,-3.337813,14.391733,-7.669085,-22.883625,-11.609543,0.177867,0.819698


In [46]:
from pathlib import Path

wav_files = list(
    Path("/content/parkinson_audio").rglob("*.wav")
)

print("Total WAV files:", len(wav_files))

for f in wav_files[:10]:
    print(f)

Total WAV files: 0


In [47]:
from pathlib import Path

dataset_path = Path("/content/Voice_Dataset")

wav_files = list(dataset_path.rglob("*.wav"))

print("Total WAV files:", len(wav_files))
print("First 5 files:")

for f in wav_files[:5]:
    print(f)

Total WAV files: 1134
First 5 files:
/content/Voice_Dataset/Voice_Dataset/Parkinsons/parkinsons_137.wav
/content/Voice_Dataset/Voice_Dataset/Parkinsons/parkinsons_530.wav
/content/Voice_Dataset/Voice_Dataset/Parkinsons/parkinsons_476.wav
/content/Voice_Dataset/Voice_Dataset/Parkinsons/parkinsons_089.wav
/content/Voice_Dataset/Voice_Dataset/Parkinsons/parkinsons_302.wav


In [48]:
!pip -q install librosa soundfile pandas numpy scikit-learn


In [49]:
import librosa
import numpy as np

sample_file = wav_files[0]

audio, sr = librosa.load(sample_file, sr=None, mono=True)

print("File:", sample_file.name)
print("Sample rate:", sr)
print("Duration:", round(len(audio) / sr, 2), "seconds")
print("Audio loaded successfully!")


File: parkinsons_137.wav
Sample rate: 8000
Duration: 3.1 seconds
Audio loaded successfully!


In [52]:
import librosa
import numpy as np
import pandas as pd
from pathlib import Path

dataset_path = Path("/content/Voice_Dataset")
audio_files = sorted(dataset_path.rglob("*.wav"))

print("Total WAV files:", len(audio_files))

def extract_features(file_path):
    y, sr = librosa.load(file_path, sr=None, mono=True)

    # Remove silent sections
    y, _ = librosa.effects.trim(y, top_db=25)

    if len(y) < sr // 2:
        raise ValueError("Audio too short after trimming")

    # MFCC features
    mfcc = librosa.feature.mfcc(
        y=y, sr=sr, n_mfcc=13
    )
    mfcc_mean = np.mean(mfcc, axis=1)

    # Spectral features
    spectral_centroid = np.mean(
        librosa.feature.spectral_centroid(y=y, sr=sr)
    )
    spectral_bandwidth = np.mean(
        librosa.feature.spectral_bandwidth(y=y, sr=sr)
    )
    zero_crossing_rate = np.mean(
        librosa.feature.zero_crossing_rate(y)
    )

    # Pitch / F0 estimate
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7"),
        sr=sr
    )

    valid_f0 = f0[np.isfinite(f0)]
    mean_f0 = np.mean(valid_f0) if len(valid_f0) else np.nan
    std_f0 = np.std(valid_f0) if len(valid_f0) else np.nan

    # Harmonic-to-noise ratio proxy
    harmonic, percussive = librosa.effects.hpss(y)
    noise_proxy = np.mean(percussive ** 2)
    harmonic_power = np.mean(harmonic ** 2)
    hnr_proxy = 10 * np.log10(
        (harmonic_power + 1e-10) / (noise_proxy + 1e-10)
    )

    result = {
        "filename": file_path.name,
        "duration_sec": len(y) / sr,
        "Pitch_mean_Hz": mean_f0,
        "Pitch_std_Hz": std_f0,
        "HNR_proxy_dB": hnr_proxy,
        "Spectral_Centroid": spectral_centroid,
        "Spectral_Bandwidth": spectral_bandwidth,
        "Zero_Crossing_Rate": zero_crossing_rate
    }

    for i, value in enumerate(mfcc_mean):
        result[f"MFCC{i}"] = value

    return result

rows = []
errors = []

for i, file_path in enumerate(audio_files, start=1):
    try:
        rows.append(extract_features(file_path))
    except Exception as e:
        errors.append((str(file_path), str(e)))

    if i % 50 == 0:
        print(f"Processed {i}/{len(audio_files)}")

features_df = pd.DataFrame(rows)
features_df.to_csv(
    "/content/voice_extracted_features.csv",
    index=False
)

print("\nSuccessful files:", len(features_df))
print("Failed files:", len(errors))
print(features_df.head())

Total WAV files: 1134
Processed 50/1134
Processed 100/1134
Processed 150/1134
Processed 200/1134
Processed 250/1134
Processed 300/1134
Processed 350/1134
Processed 400/1134
Processed 450/1134
Processed 500/1134
Processed 550/1134
Processed 600/1134
Processed 650/1134
Processed 700/1134
Processed 750/1134
Processed 800/1134
Processed 850/1134
Processed 900/1134
Processed 950/1134
Processed 1000/1134
Processed 1050/1134
Processed 1100/1134

Successful files: 1134
Failed files: 0
          filename  duration_sec  Pitch_mean_Hz  Pitch_std_Hz  HNR_proxy_dB  \
0  healthy_000.wav      3.738875     131.624739      1.893134     12.241959   
1  healthy_001.wav      3.738875     131.624739      1.893134     12.227426   
2  healthy_002.wav      3.738875     131.624739      1.893134     12.241993   
3  healthy_003.wav      3.738875     117.239779      1.662116     10.390219   
4  healthy_004.wav      3.738875     147.712775      2.087490      9.939828   

   Spectral_Centroid  Spectral_Bandwidth  Z

In [53]:
!pip install praat-parselmouth

In [54]:

import librosa
import numpy as np
import pandas as pd
import parselmouth
from parselmouth.praat import call
from pathlib import Path

# Dataset folders
root = Path("/content/Voice_Dataset")
audio_files = sorted(root.rglob("*.wav"))

print("Total audio files:", len(audio_files))

def extract_all_features(path):
    y, sr = librosa.load(path, sr=None, mono=True)
    duration = len(y) / sr

    if duration < 0.5:
        raise ValueError("Audio too short")

    # Keep original audio for voice-quality analysis
    sound = parselmouth.Sound(y, sampling_frequency=sr)

    # Trim silence for general acoustic features
    y_trim, _ = librosa.effects.trim(y, top_db=25)
    if len(y_trim) < int(sr * 0.25):
        y_trim = y

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y_trim, sr=sr, n_mfcc=13
    )
    mfcc_means = np.mean(mfcc, axis=1)

    # Spectral features
    centroid = np.mean(
        librosa.feature.spectral_centroid(y=y_trim, sr=sr)
    )
    bandwidth = np.mean(
        librosa.feature.spectral_bandwidth(y=y_trim, sr=sr)
    )
    zcr = np.mean(
        librosa.feature.zero_crossing_rate(y=y_trim)
    )

    # Pitch / F0
    f0, voiced_flag, voiced_probs = librosa.pyin(
        y_trim,
        fmin=75,
        fmax=min(600, sr / 2 - 1),
        sr=sr
    )
    valid_f0 = f0[np.isfinite(f0)]
    pitch_mean = np.mean(valid_f0) if len(valid_f0) else np.nan
    pitch_std = np.std(valid_f0) if len(valid_f0) else np.nan

    # Praat voice-quality features
    jitter = np.nan
    shimmer = np.nan
    hnr = np.nan
    voice_ratio = np.nan

    try:
        pitch_obj = call(sound, "To Pitch", 0.0, 75, 600)
        pitch_values = pitch_obj.selected_array["frequency"]
        voice_ratio = np.mean(pitch_values > 0)

        pp = call(
            sound, "To PointProcess (periodic, cc)", 75, 600
        )
        jitter = call(
            pp, "Get jitter (local)",
            0, 0, 0.0001, 0.02, 1.3
        )
        shimmer = call(
            [sound, pp], "Get shimmer (local)",
            0, 0, 0.0001, 0.02, 1.3, 1.6
        )
        harm = call(
            sound, "To Harmonicity (cc)",
            0.01, 75, 0.1, 1.0
        )
        hnr = call(harm, "Get mean", 0, 0)
    except Exception:
        pass

    # Onset rate proxy, NOT validated speech rate
    onset_frames = librosa.onset.onset_detect(
        y=y_trim, sr=sr
    )
    onset_rate_proxy = len(onset_frames) / (
        len(y_trim) / sr
    )

    row = {
        "filename": path.name,
        "duration_sec": duration,
        "Pitch_mean_Hz": pitch_mean,
        "Pitch_std_Hz": pitch_std,
        "Jitter_local": jitter,
        "Shimmer_local": shimmer,
        "HNR_dB": hnr,
        "Voice_Ratio": voice_ratio,
        "Speech_Rate_proxy": onset_rate_proxy,
        "Spectral_Centroid": centroid,
        "Spectral_Bandwidth": bandwidth,
        "Zero_Crossing_Rate": zcr,
        "Status": (
            1 if "parkinsons" in str(path).lower()
            else 0 if "healthy" in str(path).lower()
            else np.nan
        )
    }

    for i, value in enumerate(mfcc_means):
        row[f"MFCC{i}"] = value

    return row

rows = []
errors = []

for i, path in enumerate(audio_files, start=1):
    try:
        rows.append(extract_all_features(path))
    except Exception as e:
        errors.append((path.name, str(e)))

    if i % 50 == 0:
        print(f"Processed {i}/{len(audio_files)}")

features_df = pd.DataFrame(rows)

output = "/content/parkinsons_voice_features.csv"
features_df.to_csv(output, index=False)

print("\nCSV saved to:", output)
print("Successful:", len(features_df))
print("Failed:", len(errors))
print("Dataset shape:", features_df.shape)
print("\nClass counts:")
print(features_df["Status"].value_counts(dropna=False))
print("\nMissing values:")
print(features_df.isna().sum())
print("\nPreview:")
print(features_df.head())


Total audio files: 1134
Processed 50/1134
Processed 100/1134
Processed 150/1134
Processed 200/1134
Processed 250/1134
Processed 300/1134
Processed 350/1134
Processed 400/1134
Processed 450/1134
Processed 500/1134
Processed 550/1134
Processed 600/1134
Processed 650/1134
Processed 700/1134
Processed 750/1134
Processed 800/1134
Processed 850/1134
Processed 900/1134
Processed 950/1134
Processed 1000/1134
Processed 1050/1134
Processed 1100/1134

CSV saved to: /content/parkinsons_voice_features.csv
Successful: 1134
Failed: 0
Dataset shape: (1134, 26)

Class counts:
Status
0    574
1    560
Name: count, dtype: int64

Missing values:
filename              0
duration_sec          0
Pitch_mean_Hz         0
Pitch_std_Hz          0
Jitter_local          0
Shimmer_local         0
HNR_dB                0
Voice_Ratio           0
Speech_Rate_proxy     0
Spectral_Centroid     0
Spectral_Bandwidth    0
Zero_Crossing_Rate    0
Status                0
MFCC0                 0
MFCC1                 0
MFCC2 

In [55]:
from google.colab import files
files.download("/content/parkinsons_voice_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [56]:

import pandas as pd
import hashlib
from pathlib import Path

csv_path = "/content/parkinsons_voice_features.csv"
audio_root = Path("/content/Voice_Dataset")

df = pd.read_csv(csv_path)

# Check duplicate feature rows (excluding filename and label)
feature_cols = [
    c for c in df.columns
    if c not in ["filename", "Status"]
]

print("Duplicate feature rows:",
      df.duplicated(subset=feature_cols).sum())

# Check exact duplicate audio files using file hashes
hashes = {}

for path in audio_root.rglob("*.wav"):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    hashes.setdefault(digest, []).append(path.name)

duplicate_groups = [
    names for names in hashes.values() if len(names) > 1
]

print("Exact duplicate audio groups:", len(duplicate_groups))

for group in duplicate_groups[:10]:
    print(group)

print("\nClass distribution:")
print(df["Status"].value_counts())


Duplicate feature rows: 567
Exact duplicate audio groups: 567
['parkinsons_137.wav', 'parkinsons_417.wav']
['parkinsons_530.wav', 'parkinsons_250.wav']
['parkinsons_476.wav', 'parkinsons_196.wav']
['parkinsons_089.wav', 'parkinsons_369.wav']
['parkinsons_302.wav', 'parkinsons_022.wav']
['parkinsons_380.wav', 'parkinsons_100.wav']
['parkinsons_233.wav', 'parkinsons_513.wav']
['parkinsons_553.wav', 'parkinsons_273.wav']
['parkinsons_019.wav', 'parkinsons_299.wav']
['parkinsons_254.wav', 'parkinsons_534.wav']

Class distribution:
Status
0    574
1    560
Name: count, dtype: int64


In [57]:

from pathlib import Path
import hashlib
from collections import defaultdict

root = Path("/content/Voice_Dataset")
groups = defaultdict(list)

for path in root.rglob("*.wav"):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    groups[digest].append(path)

print("Unique audio contents:", len(groups))
print("Duplicate groups:", sum(len(v) > 1 for v in groups.values()))

print("\nClass labels within duplicate groups:")
cross_class = 0

for files in groups.values():
    labels = {
        "Parkinsons" if "parkinsons" in str(f).lower()
        else "Healthy" if "healthy" in str(f).lower()
        else "Unknown"
        for f in files
    }

    if len(labels) > 1:
        cross_class += 1
        print([f.name for f in files], labels)

print("Groups containing different labels:", cross_class)


Unique audio contents: 567
Duplicate groups: 567

Class labels within duplicate groups:
Groups containing different labels: 0


In [58]:
import hashlib
from pathlib import Path
import pandas as pd

audio_root = Path("/content/Voice_Dataset")
csv_path = "/content/parkinsons_voice_features.csv"

if not Path(csv_path).exists():
    raise FileNotFoundError(f"Features file {csv_path} not found.")

df = pd.read_csv(csv_path)
df["filename"] = df["filename"].astype(str)

# Group audio files by exact content
unique_filenames = []
if audio_root.exists():
    audio_groups = {}
    for path in sorted(audio_root.rglob("*.wav")):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        audio_groups.setdefault(digest, []).append(path.name)

    # Keep one file from each exact-duplicate group
    for digest, filenames in audio_groups.items():
        filenames_in_csv = [name for name in filenames if name in set(df["filename"])]
        if not filenames_in_csv:
            continue
        group_rows = df[df["filename"].isin(filenames_in_csv)]
        if group_rows["Status"].nunique() != 1:
            raise ValueError(f"Conflicting labels in duplicate group: {filenames_in_csv}")
        unique_filenames.append(sorted(filenames_in_csv)[0])

    unique_df = df[df["filename"].isin(unique_filenames)].drop_duplicates(subset="filename").copy()
else:
    print("Voice_Dataset folder not found. Keeping original features.")
    unique_df = df.copy()

unique_df.to_csv("/content/parkinsons_voice_features_unique.csv", index=False)
print("Original rows:", len(df))
print("Unique rows saved:", len(unique_df))
print("Saved: /content/parkinsons_voice_features_unique.csv")

Original rows: 1134
Unique audio contents: 567

Class counts:
Status
0    287
1    280
Name: count, dtype: int64

Missing values: 0
Saved: /content/parkinsons_voice_features_unique.csv


In [59]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import joblib

# Load unique dataset
df = pd.read_csv(
    "/content/parkinsons_voice_features_unique.csv"
)

# Prepare X and y
X = df.drop(columns=["filename", "Status"])
X = X.replace([np.inf, -np.inf], np.nan)
y = df["Status"].astype(int)

# Same test set for all 3 models
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

models = {
    "SVM": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "MLP Neural Network": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(64, 32),
            max_iter=1000,
            early_stopping=True,
            random_state=42
        ))
    ])
}

results = []

for name, model in models.items():
    print("\n" + "=" * 45)
    print("Training:", name)

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(
        y_test, predictions, zero_division=0
    )
    recall = recall_score(
        y_test, predictions, zero_division=0
    )
    f1 = f1_score(
        y_test, predictions, zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        predictions,
        target_names=["Healthy", "Parkinson's"],
        zero_division=0
    ))

    joblib.dump(
        model,
        f"/content/{name.lower().replace(' ', '_')}_model.joblib"
    )

# Compare models
results_df = pd.DataFrame(results).sort_values(
    "F1-score", ascending=False
)

print("\nMODEL COMPARISON")
print(results_df.to_string(index=False))

results_df.to_csv(
    "/content/model_comparison_results.csv",
    index=False
)

print("\nSaved model comparison and 3 trained models.")



Training: SVM
Accuracy : 0.9386
Precision: 0.9455
Recall   : 0.9286
F1-score : 0.9369

Confusion Matrix:
[[55  3]
 [ 4 52]]

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.93      0.95      0.94        58
 Parkinson's       0.95      0.93      0.94        56

    accuracy                           0.94       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.94      0.94      0.94       114


Training: Random Forest
Accuracy : 0.9561
Precision: 0.9322
Recall   : 0.9821
F1-score : 0.9565

Confusion Matrix:
[[54  4]
 [ 1 55]]

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.98      0.93      0.96        58
 Parkinson's       0.93      0.98      0.96        56

    accuracy                           0.96       114
   macro avg       0.96      0.96      0.96       114
weighted avg       0.96      0.96      0.96       114


Training: MLP Neural Network
Accuracy